# Feature Engineering — Ames Housing Dataset

## Objetivo de este notebook

Transformar las variables del dataset limpio en representaciones numéricas
que los modelos de Machine Learning puedan procesar correctamente.

**Estrategias de encoding aplicadas:**
- **Ordinal Encoding** — variables con jerarquía natural de calidad (Po < Fa < TA < Gd < Ex)
- **One-Hot Encoding** — variables nominales sin orden entre categorías
- **Binary Encoding** — variables dicotómicas (N/Y)
- **Ordinal agrupado** — Neighborhood reducido de 25 categorías a 5 segmentos por precio mediano

**Features temporales creadas:**
- `property_age = YrSold - YearBuilt`
- `year_since_remod = YrSold - YearRemodAdd`

**Input:** `data/train_clean.csv` (81 columnas, 0 nulos)
**Output:** `data/train_features.csv` (47 columnas — 46 features + 1 target)

In [ ]:
# ============================================================
# IMPORTACIÓN DE LIBRERÍAS
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import OrdinalEncoder   # Encoding para variables ordinales con jerarquía
from sklearn.preprocessing import OneHotEncoder    # Encoding para variables nominales sin orden

# Crea la carpeta images/ en la raíz del proyecto si no existe
Path('../../images').mkdir(exist_ok=True)

## Carga del Dataset Limpio

Se lee `train_clean.csv` generado en el Notebook 02. Se usa `keep_default_na=False, na_values=['']`
para evitar que los strings `"None"` codificados en la fase de missing values sean reinterpretados
como NaN por pandas — eso rompería los encoders que esperan la cadena literal `"None"`.

In [ ]:
data_path = Path('../../data/train_clean.csv')
# keep_default_na=False desactiva la interpretación automática de nulos
# na_values=[] lista vacía — ningún valor adicional se interpreta como nulo
df = pd.read_csv(data_path, keep_default_na=False, na_values=[''])

print(f'Filas: {df.shape[0]}')
print(f'Columnas: {df.shape[1]}')
print(f'Nulos totales: {df.isnull().sum().sum()}')

Filas: 1460
Columnas: 81
Nulos totales: 0


## Variables Seleccionadas para el Modelo
 
### Variables Numéricas Seleccionadas
Variables que se usan directamente sin transformación adicional.

| Variable | Descripción | Correlación con SalePrice |
|---|---|---|
| OverallQual | Calidad general de materiales y acabados (1-10) | 0.79 |
| GrLivArea | Área habitable sobre el suelo en pies cuadrados | 0.71 |
| GarageArea | Área del garaje en pies cuadrados | 0.62 |
| TotalBsmtSF | Área total del sótano en pies cuadrados | 0.61 |
| FullBath | Número de baños completos sobre el suelo | 0.56 |
| YearBuilt | Año de construcción original | 0.52 |
| YearRemodAdd | Año de última remodelación | 0.51 |
| MasVnrArea | Área de revestimiento de mampostería en pies cuadrados | 0.48 |
| Fireplaces | Número de chimeneas | 0.47 |
| BsmtFinSF1 | Área terminada del sótano tipo 1 en pies cuadrados | 0.39 |
| LotFrontage | Pies lineales de calle conectados a la propiedad | 0.35 |
| WoodDeckSF | Área de terraza de madera en pies cuadrados | 0.32 |
| 2ndFlrSF | Área del segundo piso en pies cuadrados | 0.32 |
| OpenPorchSF | Área de porche abierto en pies cuadrados | 0.32 |
| HalfBath | Número de medios baños sobre el suelo | 0.28 |

**Variables eliminadas por multicolinealidad:**
- `GarageCars` → redundante con `GarageArea`
- `TotRmsAbvGrd` → redundante con `GrLivArea`
- `GarageYrBlt` → redundante con `YearBuilt`
- `1stFlrSF` → redundante con `TotalBsmtSF`

---

### Variables Categóricas — Ordinal Encoding
Variables con jerarquía clara entre categorías. Se asigna un valor 
numérico respetando el orden de menor a mayor calidad o condición.

| Variable | Descripción | Orden |
|---|---|---|
| ExterQual | Calidad de materiales del exterior | Po < Fa < TA < Gd < Ex |
| KitchenQual | Calidad de la cocina | Po < Fa < TA < Gd < Ex |
| BsmtQual | Calidad del sótano (altura) | None < Po < Fa < TA < Gd < Ex |
| HeatingQC | Calidad del sistema de calefacción | Po < Fa < TA < Gd < Ex |
| BsmtExposure | Exposición del sótano al exterior | None < No < Mn < Av < Gd |
| BsmtFinType1 | Calidad del área terminada del sótano | None < Unf < LwQ < Rec < BLQ < ALQ < GLQ |
| GarageFinish | Acabado interior del garaje | None < Unf < RFn < Fin |
| PavedDrive | Tipo de entrada vehicular | N < P < Y |
| LotShape | Forma general del lote | IR3 < IR2 < IR1 < Reg |

---

### Variables Categóricas — One-Hot Encoding
Variables nominales sin orden jerárquico entre categorías. Se crean 
columnas binarias independientes por cada categoría para evitar 
introducir relaciones matemáticas artificiales.

| Variable | Descripción | # Categorías |
|---|---|---|
| Foundation | Tipo de cimentación | 6 |
| GarageType | Ubicación del garaje | 6 |
| MSZoning | Clasificación de zonificación general | 5 |
| SaleCondition | Condición de la venta | 6 |

---

## Variables Categóricas — Binary Encoding
Variables con exactamente 2 categorías. Se codifican como 0 y 1 
sin necesidad de One-Hot Encoding.

| Variable | Descripción | Codificación |
|---|---|---|
| CentralAir | Aire acondicionado central | N=0, Y=1 |

---

### Variables Categóricas — Ordinal Encoding Agrupado
Variables con alta cardinalidad que requieren agrupación previa 
por precio mediano antes de aplicar Ordinal Encoding.

| Variable | Descripción | # Categorías originales |
|---|---|---|
| Neighborhood | Ubicación dentro de Ames | 25 → 5 segmentos |

**Segmentos de Neighborhood por precio mediano:**
| Segmento | Valor | Vecindarios | Precio Mediano (USD) |
|---|---|---|---|
| Premium | 5 | NridgHt, NoRidge, StoneBr | 278,000 - 315,000 |
| Alto | 4 | Timber, Somerst, Veenker, Crawfor, ClearCr | 200,624 - 228,475 |
| Medio | 3 | CollgCr, Blmngtn, NWAmes, Gilbert, SawyerW | 179,900 - 197,200 |
| Bajo | 2 | Mitchel, NPkVill, NAmes, SWISU, Blueste, Sawyer | 135,000 - 153,500 |
| Muy Bajo | 1 | BrkSide, Edwards, OldTown, BrDale, IDOTRR, MeadowV | 88,000 - 124,300 |

---

### Target
| Variable | Descripción |
|---|---|
| SalePrice_log | Logaritmo natural del precio de venta — target del modelo |

## 1. Ordinal Encoding

In [ ]:
variables_calidad = ['ExterQual', 'KitchenQual', 'BsmtQual', 'HeatingQC']

# Escalas ordinales definidas según el diccionario de datos de Kaggle
# 'None' se incluye como primer nivel para variables de sótano/garaje sin la característica
orden_variables_calidad = ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
orden_bsmt_exposure = ['None', 'No', 'Mn', 'Av', 'Gd']
orden_bsmt_fin_type_1 = ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ']
orden_garage_finish = ['None', 'Unf', 'RFn', 'Fin']
orden_paved_drive = ['None', 'N', 'P', 'Y']
orden_lot_shape = ['IR3', 'IR2', 'IR1', 'Reg']

# OrdinalEncoder requiere categorías explícitas por columna y entrada 2D
# Se multiplica la lista de categorías por el número de variables que comparten la misma escala
encoder_qual = OrdinalEncoder(
    categories=[orden_variables_calidad] * len(variables_calidad)
)
encoder_bsmt_exposure  = OrdinalEncoder(categories=[orden_bsmt_exposure])
encoder_bsmt_fin_type_1= OrdinalEncoder(categories=[orden_bsmt_fin_type_1])
encoder_garage_finish  = OrdinalEncoder(categories=[orden_garage_finish])
encoder_paved_drive    = OrdinalEncoder(categories=[orden_paved_drive])
encoder_lot_shape      = OrdinalEncoder(categories=[orden_lot_shape])

# fit_transform: aprende las categorías y aplica la transformación en un solo paso
# df[[col]] (doble corchete) convierte la Serie a DataFrame 2D — requerido por OrdinalEncoder
df[variables_calidad] = encoder_qual.fit_transform(df[variables_calidad])
df['BsmtExposure'] = encoder_bsmt_exposure.fit_transform(df[['BsmtExposure']])
df['BsmtFinType1'] = encoder_bsmt_fin_type_1.fit_transform(df[['BsmtFinType1']])
df['GarageFinish'] = encoder_garage_finish.fit_transform(df[['GarageFinish']])
df['PavedDrive']   = encoder_paved_drive.fit_transform(df[['PavedDrive']])
df['LotShape']     = encoder_lot_shape.fit_transform(df[['LotShape']])

# Verificamos que todas las variables fueron codificadas correctamente
for var in variables_calidad:
    print(f'\n{var}:')
    print(df[var].value_counts().sort_index())

for var in ['BsmtExposure', 'BsmtFinType1', 'GarageFinish', 'PavedDrive', 'LotShape']:
    print(f'\n{var}:')
    print(df[var].value_counts().sort_index())

## 2. One Hot Encoding

In [50]:
variables_onehot = ['Foundation', 'GarageType', 'MSZoning', 'SaleCondition']

# drop='first' elimina la primera categoría de cada variable
# sparse_output=False retorna un array denso en lugar de matriz dispersa
# handle_unknown='ignore' ignora categorías no vistas durante el fit
encoder_onehot = OneHotEncoder(
    drop='first',
    sparse_output=False,
    handle_unknown='ignore'
)

# Aplicamos el encoding
encoded_array = encoder_onehot.fit_transform(df[variables_onehot])

# Obtenemos los nombres de las columnas generadas
columnas_nuevas = encoder_onehot.get_feature_names_out(variables_onehot)

# Creamos un DataFrame con las columnas nuevas
df_encoded = pd.DataFrame(encoded_array, columns=columnas_nuevas, index=df.index)

# Unimos al DataFrame original y eliminamos las columnas originales
df = pd.concat([df.drop(columns=variables_onehot), df_encoded], axis=1)

# Verificamos cuántas columnas tenemos ahora
print(f'Columnas totales: {df.shape[1]}')
print(f'Columnas nuevas generadas: {columnas_nuevas}')

Columnas totales: 97
Columnas nuevas generadas: ['Foundation_CBlock' 'Foundation_PConc' 'Foundation_Slab'
 'Foundation_Stone' 'Foundation_Wood' 'GarageType_Attchd'
 'GarageType_Basment' 'GarageType_BuiltIn' 'GarageType_CarPort'
 'GarageType_Detchd' 'GarageType_None' 'MSZoning_FV' 'MSZoning_RH'
 'MSZoning_RL' 'MSZoning_RM' 'SaleCondition_AdjLand'
 'SaleCondition_Alloca' 'SaleCondition_Family' 'SaleCondition_Normal'
 'SaleCondition_Partial']


In [ ]:
# Verificamos que las columnas originales fueron eliminadas correctamente
# Si alguna aparece como True, el drop no se aplicó y hay un error en el pipeline
for var in variables_onehot:
    print(f'{var} en df: {var in df.columns}')

## 3. Binary Encoding

In [52]:
# map() aplica un diccionario de mapeo a cada valor de la columna
df['CentralAir'] = df['CentralAir'].map({'N': 0, 'Y': 1})

print(df['CentralAir'].value_counts())

CentralAir
1    1365
0      95
Name: count, dtype: int64


## 4. Ordinal Encoding Agrupado

In [ ]:
# Mapeamos cada vecindario a un segmento ordinal del 1 (más barato) al 5 (más caro)
# Segmentación basada en precio mediano calculado en el EDA (Notebook 01)
# Reduce cardinalidad de 25 categorías a 5 sin perder la información de ubicación
neighborhood_segmentos = {
    'NridgHt': 5, 'NoRidge': 5, 'StoneBr': 5,          # Premium: $278K - $315K
    'Timber': 4, 'Somerst': 4, 'Veenker': 4,
    'Crawfor': 4, 'ClearCr': 4,                          # Alto: $200K - $228K
    'CollgCr': 3, 'Blmngtn': 3, 'NWAmes': 3,
    'Gilbert': 3, 'SawyerW': 3,                          # Medio: $179K - $197K
    'Mitchel': 2, 'NPkVill': 2, 'NAmes': 2,
    'SWISU': 2, 'Blueste': 2, 'Sawyer': 2,               # Bajo: $135K - $153K
    'BrkSide': 1, 'Edwards': 1, 'OldTown': 1,
    'BrDale': 1, 'IDOTRR': 1, 'MeadowV': 1              # Muy bajo: $88K - $124K
}

# map() reemplaza cada nombre de vecindario por su segmento numérico
df['Neighborhood'] = df['Neighborhood'].map(neighborhood_segmentos)
print(df['Neighborhood'].value_counts())

In [ ]:
# Distribución final de los 5 segmentos de Neighborhood
# Segmento 2 (bajo) es el más frecuente — mayoría de propiedades en zonas de precio medio-bajo
print(df['Neighborhood'].value_counts())

## 5. Nuevas Variables

In [ ]:
# property_age: antigüedad de la propiedad al momento de la venta
# Captura el efecto de depreciación física acumulada — no el año como número absoluto
df['property_age'] = df['YrSold'] - df['YearBuilt']

# year_since_remod: años desde la última remodelación al momento de la venta
# Captura el impacto de mejoras recientes en el valor percibido
df['year_since_remod'] = df['YrSold'] - df['YearRemodAdd']

# Verificamos que los valores tienen sentido (esperamos property_age ≥ 0)
print(df[['YrSold', 'YearBuilt', 'property_age']].describe())
print(df[['YrSold', 'YearRemodAdd', 'year_since_remod']].describe())

In [ ]:
# Algunas propiedades tienen YearRemodAdd > YrSold (error de datos o remodelación registrada después de la venta)
# np.maximum() clampea a 0 para evitar valores negativos sin eliminar el registro
df['year_since_remod'] = np.maximum(df['year_since_remod'], 0)

In [ ]:
# Confirmamos que el clamping eliminó todos los valores negativos
print(f'Valor mínimo year_since_remod: {df["year_since_remod"].min()}')
print(f'Valor máximo year_since_remod: {df["year_since_remod"].max()}')

## 6. Eliminar Variables Redundantes

In [ ]:
# Variables eliminadas por redundancia (identificadas en el EDA — Notebook 01)
# YearBuilt, YearRemodAdd, YrSold → reemplazadas por property_age y year_since_remod
# GarageCars → r=0.88 con GarageArea — se conserva GarageArea por ser continua y más granular
# TotRmsAbvGrd → r=0.83 con GrLivArea — se conserva GrLivArea por correlación directa con precio
# GarageYrBlt → redundante con YearBuilt (ya eliminado)
# 1stFlrSF → r≈0.82 con TotalBsmtSF — se conserva TotalBsmtSF como indicador de sótano completo
# Id → identificador de fila, no aporta información predictiva

variables_eliminar = [
    'YearBuilt', 'YearRemodAdd', 'YrSold',
    'GarageCars', 'TotRmsAbvGrd', 'GarageYrBlt',
    '1stFlrSF', 'Id'
]

df = df.drop(columns=variables_eliminar)

print(f'Columnas totales: {df.shape[1]}')
print(f'Filas totales: {df.shape[0]}')

## 7. Definición Variables Finales para el Modelo y Almacenamiento del Dataset

In [ ]:
# np.log() aplica logaritmo natural al precio original
# La inversa para predecir en USD es np.exp(predicción)
df['SalePrice_log'] = np.log(df['SalePrice'])

# Lista final de 46 features + 1 target = 47 columnas
# Orden: numéricas → ordinal → one-hot → binary → ordinal agrupado → target
variables_finales = [
    # Numéricas — usadas directamente sin transformación adicional
    'OverallQual', 'GrLivArea', 'GarageArea', 'TotalBsmtSF',
    'FullBath', 'MasVnrArea', 'Fireplaces', 'BsmtFinSF1',
    'LotFrontage', 'WoodDeckSF', '2ndFlrSF', 'OpenPorchSF',
    'HalfBath', 'property_age', 'year_since_remod',

    # Ordinal Encoding — jerarquía explícita de calidad/condición
    'ExterQual', 'KitchenQual', 'BsmtQual', 'HeatingQC',
    'BsmtExposure', 'BsmtFinType1', 'GarageFinish',
    'PavedDrive', 'LotShape',

    # One-Hot Encoding — nominales sin orden (drop=first para evitar multicolinealidad perfecta)
    'Foundation_CBlock', 'Foundation_PConc', 'Foundation_Slab',
    'Foundation_Stone', 'Foundation_Wood',
    'GarageType_Attchd', 'GarageType_Basment', 'GarageType_BuiltIn',
    'GarageType_CarPort', 'GarageType_Detchd', 'GarageType_None',
    'MSZoning_FV', 'MSZoning_RH', 'MSZoning_RL', 'MSZoning_RM',
    'SaleCondition_AdjLand', 'SaleCondition_Alloca',
    'SaleCondition_Family', 'SaleCondition_Normal',
    'SaleCondition_Partial',

    # Binary — dicotómica codificada manualmente
    'CentralAir',

    # Ordinal agrupado por precio mediano (1=muy bajo, 5=premium)
    'Neighborhood',

    # Target — logaritmo natural de SalePrice
    'SalePrice_log'
]

# Filtramos el DataFrame conservando solo las variables del modelo final
df_model = df[variables_finales]

# Guardamos el dataset final listo para modelado
FINAL_PATH = Path('../../data/train_features.csv')
df_model.to_csv(FINAL_PATH, index=False)

print(f'Dataset final guardado en: {FINAL_PATH}')
print(f'Variables: {df_model.shape[1]}')
print(f'Filas: {df_model.shape[0]}')

---

## Conclusiones — Feature Engineering

### Resumen del encoding aplicado

| Estrategia | Variables | Criterio de selección |
|------------|-----------|----------------------|
| **Ordinal Encoding** | ExterQual, KitchenQual, BsmtQual, HeatingQC, BsmtExposure, BsmtFinType1, GarageFinish, PavedDrive, LotShape (9) | Jerarquía natural de calidad/condición definida en diccionario de datos |
| **One-Hot (drop=first)** | Foundation, GarageType, MSZoning, SaleCondition (→ 20 columnas) | Nominales sin orden entre categorías; `drop=first` evita multicolinealidad perfecta |
| **Binary** | CentralAir (1) | Solo 2 categorías: N=0, Y=1 |
| **Ordinal agrupado** | Neighborhood (25→5) | Alta cardinalidad; agrupación por precio mediano preserva la señal de ubicación |
| **Features temporales** | `property_age`, `year_since_remod` | Capturan efecto de antigüedad y remodelación sin usar los años como números absolutos |

### Variables eliminadas y su justificación
- `YearBuilt`, `YearRemodAdd`, `YrSold` → reemplazadas por features temporales
- `GarageCars` → redundante con `GarageArea` (r=0.88)
- `TotRmsAbvGrd` → redundante con `GrLivArea` (r=0.83)
- `GarageYrBlt` → redundante con `YearBuilt`
- `1stFlrSF` → redundante con `TotalBsmtSF`
- `Id` → identificador de fila, no es predictor

### Resultado final
Dataset `train_features.csv` con **1,460 filas × 47 columnas** (46 features + 1 target).

### Próximo paso
**Notebook 04** — Regresión lineal como baseline para establecer el punto de referencia del modelo.